# Hiraeth — QLoRA Fine-tune on Kaggle (2x T4 / P100)

Workflow: code lives on **GitHub**, data is uploaded as a **Kaggle Dataset**, this notebook clones the repo, prepares data, smoke-tests, trains, merges, and zips the result for download.

**New here? See `docs/GETTING_STARTED.md` in the repo for a simple walkthrough. For deeper troubleshooting, see `docs/TRAINING_GUIDE.md`.**

**Before running:**
1. Settings > Accelerator > **GPU T4 x2** (or P100 x2), Internet > **On**.
2. Attach your Hiraeth Atlas Kaggle Dataset via **Add Input**.

That's it — the dataset step below auto-detects your attached file, no filenames to edit.

In [ ]:
!nvidia-smi

## 1. Clone the Hiraeth repo

In [ ]:
REPO_NAME = "Hiraeth"  # matches github.com/jadhavdurvesh/Hiraeth exactly (case-sensitive)

!git clone https://github.com/jadhavdurvesh/{REPO_NAME}.git
!ls {REPO_NAME}/scripts/

## 2. Install dependencies — with `--no-deps`

Kaggle notebooks ship with a PyTorch build matched to their CUDA driver. A plain `pip install -r requirements.txt` can silently upgrade torch and break GPU support. `--no-deps` avoids that — see `docs/TRAINING_GUIDE.md` for why.

In [ ]:
!pip install -q --no-deps -r {REPO_NAME}/scripts/requirements.txt

# bitsandbytes installed separately, always latest — it ships CUDA-version-specific
# binaries, and an old pinned version can predate Kaggle's current CUDA/PyTorch image
# (this exact failure has happened: 'No module named triton.ops' from an outdated
# bitsandbytes falling back through a broken path). See requirements.txt for details.
!pip install -q -U --no-deps bitsandbytes

import torch
print('CUDA available:', torch.cuda.is_available(), '| GPU count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(' ', i, torch.cuda.get_device_name(i))
# If CUDA available is False here, STOP and see the Troubleshooting section
# in docs/TRAINING_GUIDE.md before continuing.

## 3. Check your attached dataset

In [ ]:
!ls /kaggle/input/

## 4. Prepare the dataset

This auto-detects and converts your attached dataset's raw `.jsonl` file — nothing to edit. If you instead uploaded already-prepared `train.jsonl`/`val.jsonl` directly, you can skip this cell; Step 5 below auto-detects those too.

In [ ]:
# No need to edit anything here — prepare_dataset.py auto-detects your
# attached dataset's .jsonl file under /kaggle/input/. If you attached more
# than one dataset with a .jsonl in it, it'll list them and ask you to pass
# --input explicitly (see the printed error if that happens).

!mkdir -p /kaggle/working/data
!python {REPO_NAME}/scripts/prepare_dataset.py \
    --output_dir /kaggle/working/data \
    --val_split 0.02 \
    --system_prompt "You are Hiraeth, a helpful, precise AI assistant."

In [ ]:
import glob

# Prefer files already prepared by Step 4 above. If you uploaded pre-formatted
# train.jsonl/val.jsonl directly as your dataset instead, auto-detect those too —
# no path editing needed either way.
def find_file(name, prepared_path):
    import os
    if os.path.exists(prepared_path):
        return prepared_path
    matches = glob.glob(f'/kaggle/input/*/{name}')
    if len(matches) == 1:
        return matches[0]
    raise FileNotFoundError(
        f'Could not find {name} at {prepared_path} or uniquely under /kaggle/input/. '
        f'Found: {matches}. Set TRAIN_FILE/VAL_FILE manually if this is ambiguous.'
    )

TRAIN_FILE = find_file('train.jsonl', '/kaggle/working/data/train.jsonl')
VAL_FILE = find_file('val.jsonl', '/kaggle/working/data/val.jsonl')
print('TRAIN_FILE:', TRAIN_FILE)
print('VAL_FILE:', VAL_FILE)

## 5. Smoke test (strongly recommended before the full run)

20 steps, a few minutes. Confirms GPU detection, correct fp16/bf16 selection, and multi-GPU handling work before committing hours of GPU quota. Check the printed logs against `docs/TRAINING_GUIDE.md` Step 5.

In [ ]:
!torchrun --standalone --nproc_per_node=2 {REPO_NAME}/scripts/train.py \
    --train_file {TRAIN_FILE} \
    --val_file {VAL_FILE} \
    --output_dir /kaggle/working/hiraeth-smoketest \
    --max_steps 20

## 6. Full training run (QLoRA, sharded across both GPUs)

In [ ]:
!torchrun --standalone --nproc_per_node=2 {REPO_NAME}/scripts/train.py \
    --base_model Qwen/Qwen2.5-7B-Instruct \
    --train_file {TRAIN_FILE} \
    --val_file {VAL_FILE} \
    --output_dir /kaggle/working/hiraeth-qlora \
    --num_train_epochs 3 \
    --gradient_accumulation_steps 16 \
    --learning_rate 2e-4 \
    --max_seq_length 1024

## 7. Merge adapter into a standalone model

In [ ]:
!python {REPO_NAME}/scripts/merge_and_save.py \
    --base_model Qwen/Qwen2.5-7B-Instruct \
    --adapter_dir /kaggle/working/hiraeth-qlora \
    --output_dir /kaggle/working/hiraeth-merged

## 8. Eval spot-check (optional but recommended)

In [ ]:
!python {REPO_NAME}/scripts/run_eval.py \
    --model_dir /kaggle/working/hiraeth-merged \
    --prompts_file {REPO_NAME}/eval/eval_prompts.json \
    --output_dir /kaggle/working/eval_reports

## 9. Zip the trained model for download

`/kaggle/working` is wiped when the session ends — grab this before you close the notebook.

In [ ]:
!zip -r -q /kaggle/working/hiraeth-merged.zip /kaggle/working/hiraeth-merged
!ls -lh /kaggle/working/hiraeth-merged.zip

### Alternative: push straight to Hugging Face Hub instead of downloading a zip

A 7B model zip is ~14-15GB — often easier to push to HF Hub. Add an HF token as a Kaggle Secret (`Add-ons > Secrets`) named `HF_TOKEN`, then:

In [ ]:
# from kaggle_secrets import UserSecretsClient
# hf_token = UserSecretsClient().get_secret("HF_TOKEN")
# !huggingface-cli login --token {hf_token}
# !python {REPO_NAME}/scripts/merge_and_save.py \
#     --base_model Qwen/Qwen2.5-7B-Instruct \
#     --adapter_dir /kaggle/working/hiraeth-qlora \
#     --output_dir /kaggle/working/hiraeth-merged \
#     --push_to_hub_id YOUR_HF_USERNAME/hiraeth-7b